# 05 Task 1 Pipeline And Results

Notebook-native end-to-end Task 1 pipeline. This notebook replaces `scripts/create_task1_splits.py`, `scripts/run_task1_pipeline.py`, and `scripts/plot_results.py`.

In [ ]:
from pathlib import Path
import gc
import json
import os
import random
import shutil
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset, DataLoader, Subset
from torchvision import datasets, transforms, utils as vutils
from torchvision.models import inception_v3
from torchvision.utils import save_image

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from config import Config
from models.gan import Generator, ProjectionDiscriminator
from models.classifier import FruitCNN

ROOT


In [ ]:
DEFAULT_SIZES = [100, 200, 400, 800, 1300]
CLASS_NAMES = ["apple", "banana", "orange"]
SCENARIO_STYLE = {
    "real": {"color": "#2196F3", "marker": "o", "label": "Real only"},
    "real_aug": {"color": "#8E24AA", "marker": "D", "label": "Real + classical aug"},
    "synth": {"color": "#FF9800", "marker": "s", "label": "Synth only"},
    "both": {"color": "#4CAF50", "marker": "^", "label": "Real + Synth"},
}
SCENARIO_ORDER = ["real", "real_aug", "synth", "both"]


def materialize_file(src, dst, mode):
    src = Path(src)
    dst = Path(dst)
    if dst.exists() or dst.is_symlink():
        return
    if mode == "symlink":
        dst.symlink_to(src.resolve())
    elif mode == "hardlink":
        dst.hardlink_to(src)
    elif mode == "copy":
        shutil.copy2(src, dst)
    else:
        raise ValueError(f"Unsupported mode: {mode}")


def select_files(class_dir, n_per_class, seed):
    class_dir = Path(class_dir)
    files = sorted([p for p in class_dir.iterdir() if p.is_file()])
    if len(files) < n_per_class:
        raise ValueError(f"{class_dir} has only {len(files)} files, need {n_per_class}.")
    rng = random.Random(f"{class_dir.name}:{seed}")
    rng.shuffle(files)
    return sorted(files[:n_per_class], key=lambda p: p.name)


def build_split_notebook(train_root, out_root, n_per_class, seed=42, mode="symlink", overwrite=False):
    train_root = Path(train_root)
    out_root = Path(out_root)
    split_root = out_root / f"n{n_per_class}" / "train"
    expected_total = 0

    if overwrite and split_root.parent.exists():
        shutil.rmtree(split_root.parent)

    if split_root.parent.exists():
        existing = sum(1 for p in split_root.rglob("*") if p.is_file())
        class_count = len([p for p in train_root.iterdir() if p.is_dir()])
        if existing == n_per_class * class_count:
            return {
                "split_root": str(split_root.parent),
                "n_per_class": n_per_class,
                "total_files": existing,
                "mode": mode,
                "reused": True,
            }

    for class_dir in sorted([p for p in train_root.iterdir() if p.is_dir()], key=lambda p: p.name):
        selected = select_files(class_dir, n_per_class, seed)
        dst_dir = split_root / class_dir.name
        dst_dir.mkdir(parents=True, exist_ok=True)
        for src in selected:
            materialize_file(src, dst_dir / src.name, mode)
        expected_total += len(selected)

    return {
        "split_root": str(split_root.parent),
        "n_per_class": n_per_class,
        "total_files": expected_total,
        "mode": mode,
        "reused": False,
    }


@torch.no_grad()
def get_inception_features(images, model, device, batch_size=64):
    up = torch.nn.Upsample(size=(299, 299), mode="bilinear", align_corners=False)
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    feats = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i + batch_size].to(device)
        batch = (batch + 1) / 2
        batch = (batch - mean) / std
        feats.append(model(up(batch)).cpu())
    return torch.cat(feats, dim=0).numpy()


def calc_fid(mu1, sigma1, mu2, sigma2):
    from scipy import linalg
    diff = mu1 - mu2
    covmean = linalg.sqrtm(sigma1 @ sigma2)
    if isinstance(covmean, tuple):
        covmean = covmean[0]
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean))


def load_inception(device):
    model = inception_v3(weights="IMAGENET1K_V1", transform_input=False)
    model.fc = torch.nn.Identity()
    model.to(device)
    model.eval()
    return model


def make_gan_loader(cfg, data_root, split, train):
    tf_list = [transforms.Resize((cfg.img_size, cfg.img_size))]
    if train:
        tf_list.append(transforms.RandomHorizontalFlip())
    tf_list.extend([transforms.ToTensor(), transforms.Normalize([0.5] * 3, [0.5] * 3)])
    ds = datasets.ImageFolder(str(Path(data_root) / split), transform=transforms.Compose(tf_list))
    loader = DataLoader(ds, batch_size=cfg.gan_batch, shuffle=train, drop_last=train, **cfg.loader_options())
    return loader, ds.classes


def compute_fid(G, real_loader, num_classes, cfg, inception_model, device):
    real_imgs = []
    count = 0
    target = cfg.fid_n_samples
    for imgs, _ in real_loader:
        real_imgs.append(imgs)
        count += imgs.size(0)
        if count >= target:
            break
    real_imgs = torch.cat(real_imgs)[:target]
    n_fid = real_imgs.size(0)

    G.eval()
    fake_imgs = []
    remaining = n_fid
    while remaining > 0:
        bs = min(cfg.gan_batch, remaining)
        z = torch.randn(bs, cfg.z_dim, device=device)
        y = torch.randint(0, num_classes, (bs,), device=device)
        fake_imgs.append(G(z, y).cpu())
        remaining -= bs
    fake_imgs = torch.cat(fake_imgs)[:n_fid]
    G.train()
    real_feats = get_inception_features(real_imgs, inception_model, device)
    fake_feats = get_inception_features(fake_imgs, inception_model, device)
    mu_r, sig_r = real_feats.mean(0), np.cov(real_feats, rowvar=False)
    mu_f, sig_f = fake_feats.mean(0), np.cov(fake_feats, rowvar=False)
    return calc_fid(mu_r, sig_r, mu_f, sig_f)


def save_samples(G, fixed_z, fixed_y, epoch, out_dir, nrow=8):
    out_dir = Path(out_dir)
    G.eval()
    with torch.no_grad():
        imgs = G(fixed_z, fixed_y)
    vutils.save_image(imgs, out_dir / f"samples_epoch{epoch:04d}.png", nrow=nrow, normalize=True, value_range=(-1, 1))
    G.train()


def train_gan_for_pipeline(cfg, data_root=None, fid_root=None, out_dir=None):
    device = torch.device(cfg.resolve_device())
    data_root = Path(data_root) if data_root is not None else cfg.data_root
    fid_root = Path(fid_root) if fid_root is not None else data_root
    out_dir = Path(out_dir) if out_dir is not None else Path(cfg.out_root) / "gan"

    torch.manual_seed(cfg.seed)
    random.seed(cfg.seed)
    np.random.seed(cfg.seed)

    train_loader, class_names = make_gan_loader(cfg, data_root=data_root, split="train", train=True)
    fid_split = cfg.fid_eval_split if (Path(fid_root) / cfg.fid_eval_split).exists() else "train"
    fid_loader, fid_classes = make_gan_loader(cfg, data_root=fid_root, split=fid_split, train=False)
    if fid_classes != class_names:
        raise RuntimeError(f"Class mismatch: {class_names} vs {fid_classes}")
    num_classes = len(class_names)

    G = Generator(z_dim=cfg.z_dim, num_classes=num_classes).to(device)
    D = ProjectionDiscriminator(num_classes=num_classes).to(device)
    criterion = nn.BCEWithLogitsLoss()
    opt_G = torch.optim.Adam(G.parameters(), lr=cfg.gan_lr_g, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=cfg.gan_lr_d, betas=(0.5, 0.999))
    fixed_z = torch.randn(24, cfg.z_dim, device=device)
    fixed_y = torch.arange(num_classes, device=device).repeat_interleave(8)

    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir = out_dir / "checkpoints"
    ckpt_dir.mkdir(exist_ok=True)
    inception_model = load_inception(device)
    fid_log = []
    best_fid = float("inf")
    best_epoch = None
    start_time = time.time()

    for epoch in range(1, cfg.gan_epochs + 1):
        d_loss_acc = g_loss_acc = d_x_acc = d_gz_acc = 0.0
        n_batches = 0
        for real_imgs, real_labels in train_loader:
            real_imgs = real_imgs.to(device)
            real_labels = real_labels.to(device)
            batch_size = real_imgs.size(0)
            real_targets = torch.ones(batch_size, 1, device=device)
            fake_targets = torch.zeros(batch_size, 1, device=device)

            with torch.no_grad():
                z = torch.randn(batch_size, cfg.z_dim, device=device)
                fake_imgs = G(z, real_labels)
            d_real_out = D(real_imgs, real_labels)
            d_fake_out = D(fake_imgs, real_labels)
            d_loss = criterion(d_real_out, real_targets) + criterion(d_fake_out, fake_targets)
            opt_D.zero_grad()
            d_loss.backward()
            nn.utils.clip_grad_norm_(D.parameters(), max_norm=1.0)
            opt_D.step()

            z = torch.randn(batch_size, cfg.z_dim, device=device)
            gen_labels = torch.randint(0, num_classes, (batch_size,), device=device)
            fake_imgs = G(z, gen_labels)
            g_out = D(fake_imgs, gen_labels)
            g_loss = criterion(g_out, torch.ones(batch_size, 1, device=device))
            opt_G.zero_grad()
            g_loss.backward()
            nn.utils.clip_grad_norm_(G.parameters(), max_norm=1.0)
            opt_G.step()

            d_loss_acc += d_loss.item()
            g_loss_acc += g_loss.item()
            d_x_acc += torch.sigmoid(d_real_out).mean().item()
            d_gz_acc += torch.sigmoid(g_out).mean().item()
            n_batches += 1

        d_avg = d_loss_acc / n_batches
        g_avg = g_loss_acc / n_batches
        d_x_avg = d_x_acc / n_batches
        d_gz_avg = d_gz_acc / n_batches
        entry = {"epoch": epoch, "d_loss": d_avg, "g_loss": g_avg, "d_x": d_x_avg, "d_gz": d_gz_avg}
        if epoch % cfg.fid_every == 0 or epoch == cfg.gan_epochs:
            fid = compute_fid(G, fid_loader, num_classes, cfg, inception_model, device)
            entry["fid"] = fid
            if fid < best_fid:
                best_fid = fid
                best_epoch = epoch
                torch.save({"epoch": epoch, "fid": fid, "G": G.state_dict(), "D": D.state_dict(), "opt_G": opt_G.state_dict(), "opt_D": opt_D.state_dict()}, ckpt_dir / "best_fid.pt")
        fid_log.append(entry)
        if epoch % cfg.sample_every == 0 or epoch == 1:
            save_samples(G, fixed_z, fixed_y, epoch, out_dir)
        if epoch % cfg.ckpt_every == 0 or epoch == cfg.gan_epochs:
            torch.save({"epoch": epoch, "G": G.state_dict(), "D": D.state_dict(), "opt_G": opt_G.state_dict(), "opt_D": opt_D.state_dict()}, ckpt_dir / f"ckpt_epoch{epoch:04d}.pt")

    with open(out_dir / "train_log.json", "w") as f:
        json.dump(fid_log, f, indent=2)
    summary = {
        "data_root": str(data_root),
        "fid_root": str(fid_root),
        "fid_split": fid_split,
        "num_classes": num_classes,
        "train_samples": len(train_loader.dataset),
        "fid_samples": len(fid_loader.dataset),
        "train_time_sec": round(time.time() - start_time, 1),
        "best_fid": None if best_fid == float("inf") else round(best_fid, 4),
        "best_epoch": best_epoch,
        "out_dir": str(out_dir),
    }
    with open(out_dir / "run_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    return {"G": G, "D": D, "summary": summary}


def generate_synth_pool_notebook(ckpt, n_per_class, out_dir, batch_size=64, seed=42, class_names=None):
    ckpt = Path(ckpt)
    out_root = Path(out_dir)
    class_names = class_names or CLASS_NAMES
    cfg = Config()
    device = torch.device(cfg.resolve_device())
    G = Generator(z_dim=cfg.z_dim, num_classes=len(class_names)).to(device)
    ckpt_state = torch.load(ckpt, map_location=device, weights_only=True)
    G.load_state_dict(ckpt_state["G"])
    G.eval()
    torch.manual_seed(seed)
    start_time = time.time()
    counts = {}
    for cls_idx, cls_name in enumerate(class_names):
        cls_dir = out_root / cls_name
        cls_dir.mkdir(parents=True, exist_ok=True)
        generated = 0
        while generated < n_per_class:
            bs = min(batch_size, n_per_class - generated)
            z = torch.randn(bs, cfg.z_dim, device=device)
            y = torch.full((bs,), cls_idx, dtype=torch.long, device=device)
            with torch.no_grad():
                imgs = G(z, y)
            for i in range(bs):
                save_image(imgs[i], cls_dir / f"{cls_name}_synth_{generated + i:05d}.png", normalize=True, value_range=(-1, 1))
            generated += bs
        counts[cls_name] = generated
    summary = {
        "checkpoint": str(ckpt),
        "out_dir": str(out_root),
        "n_per_class": n_per_class,
        "counts": counts,
        "seed": seed,
        "generate_time_sec": round(time.time() - start_time, 1),
    }
    with open(out_root / "generation_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    return summary


def get_transform(img_size, train=True, augmentation_policy="none"):
    tf_list = [transforms.Resize((img_size, img_size))]
    if train and augmentation_policy == "classical":
        tf_list.extend([transforms.RandomHorizontalFlip(), transforms.RandomRotation(15), transforms.ColorJitter(brightness=0.2, contrast=0.2)])
    tf_list.extend([transforms.ToTensor(), transforms.Normalize([0.5] * 3, [0.5] * 3)])
    return transforms.Compose(tf_list)


def scenario_augmentation_policy(scenario):
    return "classical" if scenario == "real_aug" else "none"


def subsample_dataset(dataset, n_per_class, seed=42):
    from collections import defaultdict
    rng = random.Random(seed)
    class_indices = defaultdict(list)
    for idx, (_, label) in enumerate(dataset.samples):
        class_indices[label].append(idx)
    selected = []
    for label, indices in sorted(class_indices.items()):
        rng.shuffle(indices)
        selected.extend(indices[:n_per_class])
    return Subset(dataset, selected)


def build_dataset(cfg, scenario, n_per_class, synth_dir="data_synth", real_train_root=None, test_root=None):
    augmentation_policy = scenario_augmentation_policy(scenario)
    tf_train = get_transform(cfg.img_size, train=True, augmentation_policy=augmentation_policy)
    tf_test = get_transform(cfg.img_size, train=False, augmentation_policy="none")
    real_train_root = Path(real_train_root) if real_train_root else cfg.data_root / "train"
    test_root = Path(test_root) if test_root else cfg.data_root / "test"
    real_train = datasets.ImageFolder(str(real_train_root), transform=tf_train)
    test_ds = datasets.ImageFolder(str(test_root), transform=tf_test)
    if scenario in {"real", "real_aug"}:
        train_ds = subsample_dataset(real_train, n_per_class, cfg.seed) if n_per_class else real_train
    elif scenario == "synth":
        synth_train = datasets.ImageFolder(str(synth_dir), transform=tf_train)
        train_ds = subsample_dataset(synth_train, n_per_class, cfg.seed) if n_per_class else synth_train
    elif scenario == "both":
        synth_train = datasets.ImageFolder(str(synth_dir), transform=tf_train)
        if n_per_class:
            train_ds = ConcatDataset([subsample_dataset(real_train, n_per_class, cfg.seed), subsample_dataset(synth_train, n_per_class, cfg.seed)])
        else:
            train_ds = ConcatDataset([real_train, synth_train])
    else:
        raise ValueError(f"Unknown scenario: {scenario}")
    return train_ds, test_ds, real_train.classes, augmentation_policy


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
    return all_preds, all_labels


def run_classifier_notebook(cfg, scenario, n_per_class, synth_dir, out_dir, real_train_root=None, test_root=None, time_breakdown=None, extra_metadata=None):
    from sklearn.metrics import classification_report
    device = torch.device(cfg.resolve_device())
    torch.manual_seed(cfg.seed)
    random.seed(cfg.seed)
    train_ds, test_ds, class_names, augmentation_policy = build_dataset(cfg, scenario, n_per_class, synth_dir=synth_dir, real_train_root=real_train_root, test_root=test_root)
    loader_kwargs = cfg.loader_options()
    train_loader = DataLoader(train_ds, batch_size=cfg.clf_batch, shuffle=True, drop_last=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, batch_size=cfg.clf_batch, shuffle=False, **loader_kwargs)
    model = FruitCNN(num_classes=len(class_names)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.clf_lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.clf_epochs)
    t0 = time.time()
    for epoch in range(1, cfg.clf_epochs + 1):
        train_one_epoch(model, train_loader, criterion, optimizer, device)
        scheduler.step()
    train_time = time.time() - t0
    preds, labels = evaluate(model, test_loader, device)
    report = classification_report(labels, preds, target_names=class_names, output_dict=True)
    time_breakdown = time_breakdown or {}
    gan_train_time = round(float(time_breakdown.get("gan_train_time_sec", 0.0)), 1)
    synth_generation_time = round(float(time_breakdown.get("synth_generation_time_sec", 0.0)), 1)
    result = {
        "scenario": scenario,
        "augmentation_policy": augmentation_policy,
        "n_per_class": n_per_class,
        "train_size": len(train_ds),
        "test_accuracy": round(report["accuracy"], 4),
        "train_time_sec": round(train_time, 1),
        "classifier_train_time_sec": round(train_time, 1),
        "gan_train_time_sec": gan_train_time,
        "synth_generation_time_sec": synth_generation_time,
        "pipeline_time_sec": round(train_time + gan_train_time + synth_generation_time, 1),
        "per_class": {name: {"precision": round(report[name]["precision"], 4), "recall": round(report[name]["recall"], 4), "f1": round(report[name]["f1-score"], 4)} for name in class_names},
    }
    if extra_metadata:
        result.update(extra_metadata)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / f"result_{scenario}_n{n_per_class}.json", "w") as f:
        json.dump(result, f, indent=2)
    return result


def scenario_time_breakdown(scenario, gan_summary, synth_summary):
    if scenario not in {"synth", "both"}:
        return {}
    return {
        "gan_train_time_sec": gan_summary.get("train_time_sec", 0.0),
        "synth_generation_time_sec": synth_summary.get("generate_time_sec", 0.0),
    }


def run_task1_pipeline_notebook(cfg, sizes=None, train_root=None, fid_root=None, test_root=None, split_root=None, out_root=None, split_mode="symlink", overwrite_splits=False, include_real_aug=True, synth_batch_size=64):
    sizes = sizes or DEFAULT_SIZES
    train_root = Path(train_root or ROOT / "data_final" / "train")
    fid_root = Path(fid_root or ROOT / "data_final")
    test_root = Path(test_root or ROOT / "data_final" / "test")
    split_root = Path(split_root or ROOT / "data_splits" / "task1")
    out_root = Path(out_root or ROOT / "runs" / "task1")
    clf_out_dir = out_root / "clf"
    scenarios = ["real", "synth", "both"] + (["real_aug"] if include_real_aug else [])

    all_results = []
    pipeline_summary = []
    for n in sizes:
        split_summary = build_split_notebook(train_root, split_root, n, seed=cfg.seed, mode=split_mode, overwrite=overwrite_splits)
        split_root_n = Path(split_summary["split_root"])
        real_train_root = split_root_n / "train"
        gan_out_dir = out_root / "gan" / f"n{n}"
        gan_run = train_gan_for_pipeline(cfg, data_root=split_root_n, fid_root=fid_root, out_dir=gan_out_dir)
        gan_summary = gan_run["summary"]
        del gan_run
        gc.collect()
        best_ckpt = Path(gan_summary["out_dir"]) / "checkpoints" / "best_fid.pt"
        synth_out_dir = out_root / "synth" / f"n{n}"
        synth_summary = generate_synth_pool_notebook(best_ckpt, n, synth_out_dir, batch_size=synth_batch_size, seed=cfg.seed)
        size_summary = {"n_per_class": n, "split_root": str(split_root_n), "gan": gan_summary, "synth": synth_summary, "classifier_results": []}
        for scenario in scenarios:
            result = run_classifier_notebook(
                cfg,
                scenario,
                n,
                str(synth_out_dir),
                str(clf_out_dir),
                real_train_root=str(real_train_root),
                test_root=str(test_root),
                time_breakdown=scenario_time_breakdown(scenario, gan_summary, synth_summary),
                extra_metadata={
                    "generator_train_n_per_class": n,
                    "synthetic_pool_n_per_class": n,
                    "task1_fair_pipeline": True,
                },
            )
            all_results.append(result)
            size_summary["classifier_results"].append(result)
        pipeline_summary.append(size_summary)

    clf_out_dir.mkdir(parents=True, exist_ok=True)
    with open(clf_out_dir / "all_results.json", "w") as f:
        json.dump(all_results, f, indent=2)
    out_root.mkdir(parents=True, exist_ok=True)
    with open(out_root / "pipeline_summary.json", "w") as f:
        json.dump(pipeline_summary, f, indent=2)
    return all_results, pipeline_summary


def group_by_scenario(results):
    grouped = {}
    for r in results:
        grouped.setdefault(r["scenario"], []).append(r)
    for runs in grouped.values():
        runs.sort(key=lambda x: x["n_per_class"])
    return {k: grouped[k] for k in SCENARIO_ORDER if k in grouped}


def time_value(result, time_field="pipeline_time_sec"):
    if time_field == "pipeline_time_sec":
        return result.get("pipeline_time_sec", result.get("train_time_sec", 0.0))
    if time_field == "classifier_train_time_sec":
        return result.get("classifier_train_time_sec", result.get("train_time_sec", 0.0))
    return result.get(time_field, result.get("train_time_sec", 0.0))


def plot_results_notebook(results, out_dir, time_field="pipeline_time_sec"):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    grouped = group_by_scenario(results)

    fig, ax = plt.subplots(figsize=(8, 5))
    for scenario, runs in grouped.items():
        style = SCENARIO_STYLE[scenario]
        ax.plot([r["n_per_class"] for r in runs], [r["test_accuracy"] * 100 for r in runs], marker=style["marker"], color=style["color"], label=style["label"], linewidth=2, markersize=8)
    ax.set_xlabel("Training images per class")
    ax.set_ylabel("Test Accuracy (%)")
    ax.set_title("Classification Accuracy vs Training Data Size")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_dir / "accuracy_vs_size.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 5))
    for scenario, runs in grouped.items():
        style = SCENARIO_STYLE[scenario]
        ax.plot([r["n_per_class"] for r in runs], [time_value(r, time_field) for r in runs], marker=style["marker"], color=style["color"], label=style["label"], linewidth=2, markersize=8)
    ax.set_xlabel("Training images per class")
    ax.set_ylabel("Time (seconds)")
    ax.set_title("Pipeline Cost vs Training Data Size" if time_field == "pipeline_time_sec" else "Training Time vs Data Size")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_dir / "time_vs_size.png", dpi=150)
    plt.close(fig)

    fig, axes = plt.subplots(1, len(grouped), figsize=(4.5 * max(1, len(grouped)), 4), sharey=True)
    if len(grouped) == 1:
        axes = [axes]
    for ax, (scenario, runs) in zip(axes, grouped.items()):
        r = runs[-1]
        classes = list(r["per_class"].keys())
        f1s = [r["per_class"][c]["f1"] * 100 for c in classes]
        ax.bar(classes, f1s, color=["#EF5350", "#FFEE58", "#FFA726"], edgecolor="black", linewidth=0.5)
        ax.set_title(f"{SCENARIO_STYLE[scenario]['label']}\n(n={r['n_per_class']}/class)")
        ax.set_ylim(0, 105)
        if ax == axes[0]:
            ax.set_ylabel("F1 Score (%)")
    fig.tight_layout()
    fig.savefig(out_dir / "per_class_f1.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return grouped


In [ ]:
# Edit these values before running.
cfg = Config()
SIZES = DEFAULT_SIZES
TRAIN_ROOT = ROOT / "data_final" / "train"
FID_ROOT = ROOT / "data_final"
TEST_ROOT = ROOT / "data_final" / "test"
SPLIT_ROOT = ROOT / "data_splits" / "task1"
OUT_ROOT = ROOT / "runs" / "task1"
INCLUDE_REAL_AUG = True

cfg, SIZES, OUT_ROOT


In [ ]:
# Run this when you want the full pipeline.
# all_results, pipeline_summary = run_task1_pipeline_notebook(
#     cfg,
#     sizes=SIZES,
#     train_root=TRAIN_ROOT,
#     fid_root=FID_ROOT,
#     test_root=TEST_ROOT,
#     split_root=SPLIT_ROOT,
#     out_root=OUT_ROOT,
#     include_real_aug=INCLUDE_REAL_AUG,
# )


In [ ]:
results_path = OUT_ROOT / "clf" / "all_results.json"
pipeline_summary_path = OUT_ROOT / "pipeline_summary.json"

if results_path.exists():
    results = json.loads(results_path.read_text())
    print(f"Loaded {len(results)} classifier results")
    results[:2]
else:
    print("No Task 1 results yet. Run the pipeline cell first.")


In [ ]:
if results_path.exists():
    grouped = plot_results_notebook(results, OUT_ROOT / "clf" / "plots", time_field="pipeline_time_sec")
    list(grouped.keys())
